In [1]:
import json
import os

os.chdir("..")

In [2]:
from cluster_intrep_repo.utils import DOMAIN_PHRASES

In [3]:
# def make_mean_reprs_layer(layer: int):
all_representations = {}

for i in range(1, 16):
    with open(f"multilayer_representations/multilayer_7k/mystery_{i}/mean_reprs_mystery_{i}_multi_layer.json") as f:
        all_representations[f"mystery_{i}"] = json.load(f)

In [13]:
import numpy as np

avg_representations = {k: {} for k in all_representations.keys()}

def process_layer(layer: int):
    representations = {
        k: v[f"{layer}"] for k, v in all_representations.items()
    }
    
    action_reprs = {
        k: {
            kk: np.array(vv) - np.array(v["mean_actions"]) for kk, vv in v["mean_reprs"].items() if kk in DOMAIN_PHRASES[k]["actions"].values()
        } for k, v in representations.items()
    }

    action_means = {
        k: v["mean_actions"] for k, v in representations.items()
    }

    domain_action_mean = np.mean(np.stack([action_means[k] for k in action_means]), axis=0)

    predicate_reprs = {
        k: {
            kk: np.array(vv) - np.array(v["mean_predicates"]) for kk, vv in v["mean_reprs"].items() if kk in DOMAIN_PHRASES[k]["predicates"].values()
        } for k, v in representations.items()
    }

    predicate_means = {
        k: v["mean_predicates"] for k, v in representations.items()
    }

    domain_predicate_mean = np.mean(np.stack([predicate_means[k] for k in predicate_means]), axis=0)

    # action_reprs = {
    #     k: {
    #         kk: np.array(vv) for kk, vv in v["mean_reprs"].items() if kk in DOMAIN_PHRASES[k]["actions"].values()
    #     } for k, v in representations.items()
    # }

    # predicate_reprs = {
    #     k: {
    #         kk: np.array(vv) for kk, vv in v["mean_reprs"].items() if kk in DOMAIN_PHRASES[k]["predicates"].values()
    #     } for k, v in representations.items()
    # }
    
    reverse_phrases = {
        k: {kk: {
            vvv: kkk for kkk, vvv in vv.items()
        } for kk, vv in v.items() }
        for k, v in DOMAIN_PHRASES.items()
    }
    
    predicates = list(reverse_phrases["mystery_3"]["predicates"].values())
    actions = list(reverse_phrases["mystery_3"]["actions"].values())
    
    action_reprs = {
        k: {
            reverse_phrases[k]["actions"][kk]: vv for kk, vv in v.items()
        } for k, v in action_reprs.items()
    }

    predicate_reprs = {
        k: {
            reverse_phrases[k]["predicates"][kk]: vv for kk, vv in v.items()
        } for k, v in predicate_reprs.items()
    }
    
    mean_action_reprs = {
        action: np.mean(
            np.stack(
                [action_reprs[m][action] for m in action_reprs]
            ), axis=0
        ) for action in actions
    }
    
    mean_predicate_reprs = {
        pr: np.mean(
            np.stack(
                [predicate_reprs[m][pr] for m in predicate_reprs]
            ), axis=0
        ) for pr in predicates
    }
    
    for i in range(1, 16):
        reprs = {
            "mean_domain": np.mean(np.stack([domain_action_mean, domain_predicate_mean]), axis=0).tolist(),
            "mean_actions": domain_action_mean.tolist(),
            "mean_predicates": domain_predicate_mean.tolist(),
            "mean_reprs": {
                DOMAIN_PHRASES[f"mystery_{i}"]["actions"][action]: mean_action_reprs[action].tolist() for action in actions
            }
        }
        
        reprs["mean_reprs"].update({
            DOMAIN_PHRASES[f"mystery_{i}"]["predicates"][predicate]: mean_predicate_reprs[predicate].tolist() for predicate in predicates
        })
            
        avg_representations[f"mystery_{i}"][f"{layer}"] = reprs

In [14]:
from tqdm import tqdm

for i in tqdm(range(50)):
    process_layer(i)

100%|██████████| 50/50 [00:03<00:00, 15.96it/s]


In [15]:
np.linalg.norm(avg_representations["mystery_1"]["2"]["mean_reprs"]["attack"])

3.393680266855925

In [16]:
from pathlib import Path

for i in range(1, 16):
    path = Path(f"multilayer_representations_avg_new/multilayer_7k/mystery_{i}/mean_reprs_mystery_{i}_multi_layer.json")
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(avg_representations[f"mystery_{i}"], f, indent=4)